<a href="https://colab.research.google.com/github/varadarajansv-hub/Home-Solar-Automation-based-on-AI/blob/main/Home_solar_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
python

import numpy as np

np.random.seed(8)  # Seed for repeatable local weather profiles

class HomeSolarOptimization:
    def __init__(self, billing_cycles=1):
        # 1 Billing Cycle = 60 Days (Bi-monthly billing)
        self.days = billing_cycles * 60
        self.hours = 24
        self.total_steps = self.days * self.hours

        # System Configuration
        self.solar_capacity_kwp = 3.0     # 3 kWp Solar Rooftop Array - Max allowed for receiving subsidy
        self.battery_capacity_kwh = 5.0    # 5 kWh Residential Battery Storage - Assumed capacity
        self.battery_soc = 2.0             # Initial state of charge (kWh)
        self.battery_efficiency = 0.88     # Round-trip efficiency factor

    def get_DISCOM_base_rate(self, total_units_bimonthly):
        """Returns the marginal tariff rate based on cumulative bimonthly consumption."""
        # DISCOM progressive domestic tier structure (Bi-monthly units breakdown)
        if total_units_bimonthly <= 100:
            return 0.00   # Free allowance tier
        elif total_units_bimonthly <= 400:
            return 4.60   # Mid-tier base rate (₹/unit)
        elif total_units_bimonthly <= 500:
            return 6.15   # Upper mid-tier base rate
        elif total_units_bimonthly <= 600:
            return 8.55   # High-consumption tier 1
        elif total_units_bimonthly <= 800:
            return 9.65   # High-consumption tier 2
        else:
            return 11.80  # Maximum domestic tier (Above 1000 units cumulative)

    def simulate_local_profiles(self):
        """Simulates local solar radiation profile and home (usage) load curves."""
        # Base Indian household load curve with heavy evening AC/appliance spike, 24 hours basis
        base_load = np.array([0.5, 0.4, 0.4, 0.3, 0.3, 0.5, 0.8, 1.0, 0.7, 0.6, 0.5, 0.5,
                              0.6, 0.6, 0.5, 0.5, 0.7, 1.1, 2.2, 2.6, 2.3, 1.6, 1.0, 0.6])
        self.load_profile = np.tile(base_load, self.days)

        # Thermal Solar Integration
        # Solar Water Heaters shave morning loads (06:00-09:00)
        # Solar Cookers shave heavy afternoon cooking loads (11:00-14:00)
        thermal_shave = np.array([0,0,0,0,0,0,0.4,0.5,0.3,0,0,0.5,0.6,0.6,0,0,0,0,0,0,0,0,0,0])
        self.load_profile -= np.tile(thermal_shave, self.days)
        self.load_profile = np.clip(self.load_profile, 0.1, None)

        # Solar Generation profile matching typical clear/intermittent sky conditions
        base_solar = np.array([0,0,0,0,0,0,0,0.15,0.5,1.2,2.1,2.8,3.0,2.9,2.2,1.3,0.4,0.05,0,0,0,0,0,0])
        self.solar_profile = np.tile(base_solar, self.days)

        # Introduce Northeast & Southwest Monsoon patterns (Random drop drops in generation)
        for d in range(self.days):
            weather_anomaly = np.random.choice([1.0, 0.8, 0.4, 0.15], p=[0.6, 0.2, 0.15, 0.05])
            idx = d * 24
            self.solar_profile[idx:idx+24] *= weather_anomaly

    def calculate_tod_multiplier(self, hour):
        """Applies Time-of-Day (ToD) weighting factors on the base energy charges."""
        if 9 <= hour < 17:
            return 0.80  # Solar Window: 20% Discount
        elif (6 <= hour < 10) or (18 <= hour < 22):
            return 1.15  # Peak Windows: 15% Surcharge
        else:
            return 1.00  # Standard off-peak hours

    def execute_ai_heuristic_logic(self):
        self.simulate_local_profiles()

        cumulative_imported_units = 0.0
        cumulative_exported_units = 0.0
        total_billing_cost = 0.0

        for step in range(self.total_steps):
            hour = step % 24
            current_load = self.load_profile[step]
            current_solar = self.solar_profile[step]

            # Dynamic dynamic calculation based on progressive running total
            base_rate = self.get_discom_base_rate(cumulative_imported_units)
            tod_multiplier = self.calculate_tod_multiplier(hour)
            current_tariff = base_rate * tod_multiplier

            net_energy = current_solar - current_load

            if net_energy > 0:
                # Surplus solar window scenario
                room_in_battery = self.battery_capacity_kwh - self.battery_soc
                charge_power = min(net_energy, room_in_battery)
                self.battery_soc += charge_power * self.battery_efficiency

                # Discom injection (credited at lower net-metering baseline of ~₹3.10/unit)
                excess_to_grid = net_energy - charge_power
                cumulative_exported_units += excess_to_grid
                total_billing_cost -= excess_to_grid * 3.10
            else:
                # Deficit energy requirement scenario
                required_deficit = abs(net_energy)

                # AI Heuristic Rule: Only discharge battery during standard or peak tariff windows.
                # If it's a cheap solar window or night, allow DISCOM power to preserve battery life.
                if tod_multiplier >= 1.0:
                    discharge_power = min(required_deficit, self.battery_soc)
                    self.battery_soc -= discharge_power
                    grid_draw = required_deficit - (discharge_power * self.battery_efficiency)
                else:
                    grid_draw = required_deficit

                cumulative_imported_units += grid_draw
                total_billing_cost += grid_draw * current_tariff

        return {
            "Total Evaluation Period (Days)": self.days,
            "Total Units Imported from DISCOM (kWh)": round(cumulative_imported_units, 1),
            "Total Units Exported to DISCOM (kWh)": round(cumulative_exported_units, 1),
            "Final Estimated DISCOM Bill (INR)": round(max(0, total_billing_cost), 2),
            "End-of-Cycle Battery Reserve (kWh)": round(self.battery_soc, 2)
        }

# Execute the simulation block
tn_simulation = HomeSolarOptimization(billing_cycles=1)
analysis_results = local_simulation.execute_ai_heuristic_logic()

for key, metric in analysis_results.items():
    print(f"{key}: {metric}")

